# 02 — Data Exploration

Explores the curated dataset — source breakdown, text length distributions, quality filter rejection rates, deduplication statistics, and token estimates.

Run after `python curator/scripts/curate.py --target 125m --stage blend`.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import random
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

DATA_DIR = Path('../data')
CURATED_DIR = DATA_DIR / 'curated'
RAW_DIR = DATA_DIR / 'raw'
FILTERED_DIR = DATA_DIR / 'filtered'

## 1. Blend stats overview

In [ ]:
stats_path = CURATED_DIR / 'blend_stats.json'
with open(stats_path) as f:
    stats = json.load(f)

print(f"Target model:       {stats['target']}")
print(f"Total documents:    {stats['total_documents']:,}")
print(f"Total chars:        {stats['total_chars'] / 1e9:.2f}B")
print(f"Estimated tokens:   {stats['estimated_tokens'] / 1e9:.2f}B")
print()
print("Source mix:")
for source, count in stats['source_mix'].items():
    pct = 100 * count / stats['total_documents']
    print(f"  {source:<20} {count:>10,}  ({pct:.1f}%)")

## 2. Source distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sources = list(stats['source_mix'].keys())
counts = list(stats['source_mix'].values())
colors = ['#4C72B0', '#55A868', '#DD8452']

wedges, texts, autotexts = ax.pie(
    counts, labels=sources, autopct='%1.1f%%',
    colors=colors, startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for t in autotexts:
    t.set_fontsize(10)

ax.set_title('Source Distribution — Final Blend', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Load a sample of the curated dataset

In [ ]:
SAMPLE_SIZE = 10_000
train_path = CURATED_DIR / 'train.jsonl'

records = []
with open(train_path) as f:
    for i, line in enumerate(f):
        if i >= SAMPLE_SIZE:
            break
        records.append(json.loads(line))

print(f"Loaded {len(records):,} records for analysis")
print(f"Keys: {list(records[0].keys())}")

## 4. Text length distribution

In [ ]:
lengths = {source: [] for source in ['wikipedia', 'code', 'common_crawl']}
for r in records:
    src = r.get('source', 'unknown')
    if src in lengths:
        lengths[src].append(len(r['text']))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'wikipedia': '#4C72B0', 'code': '#55A868', 'common_crawl': '#DD8452'}

for ax, (source, lens) in zip(axes, lengths.items()):
    if not lens:
        ax.set_title(f'{source}\n(no data)')
        continue
    ax.hist(lens, bins=50, color=colors[source], edgecolor='white', alpha=0.85)
    ax.set_title(f'{source}\nn={len(lens):,}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
    ax.axvline(np.median(lens), color='red', linestyle='--', label=f'Median: {np.median(lens):.0f}')
    ax.legend(fontsize=9)

fig.suptitle('Text Length Distribution by Source', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary stats
rows = []
for source, lens in lengths.items():
    if lens:
        rows.append({
            'Source': source,
            'Count': len(lens),
            'Min': min(lens),
            'Median': int(np.median(lens)),
            'Mean': int(np.mean(lens)),
            'Max': max(lens),
            'p95': int(np.percentile(lens, 95)),
        })
print(pd.DataFrame(rows).to_string(index=False))

## 5. Quality filter simulation

In [ ]:
from curator.filters.quality import QualityFilter

qf = QualityFilter()
qf.reset_stats()

# Simulate on raw data sample (load from raw if available)
raw_cc = RAW_DIR / 'common_crawl'
if raw_cc.exists():
    raw_records = []
    for shard in sorted(raw_cc.glob('*.jsonl'))[:2]:  # first 2 shards
        with open(shard) as f:
            for i, line in enumerate(f):
                if i >= 5000:
                    break
                raw_records.append(json.loads(line))

    for r in raw_records:
        qf.check(r)

    print(qf.report())
else:
    print("Raw Common Crawl data not found — run curator first")
    print("Simulating on curated sample instead...")
    cc_records = [r for r in records if r.get('source') == 'common_crawl']
    for r in cc_records:
        qf.check(r)
    print(qf.report())

## 6. Token estimate

In [ ]:
# Estimate tokens using chars/4 proxy and compare with actual tokenizer if available
total_chars = sum(len(r['text']) for r in records)
estimated_tokens = total_chars // 4

print(f"Sample size:           {len(records):,} documents")
print(f"Total chars (sample):  {total_chars:,}")
print(f"Est. tokens (sample):  {estimated_tokens:,}")
print(f"Avg chars/doc:         {total_chars // len(records):,}")
print(f"Avg tokens/doc (est):  {estimated_tokens // len(records):,}")

# Try actual tokenizer if available
tokenizer_path = Path('../data/tokenizer/slm_tokenizer.json')
if tokenizer_path.exists():
    from tokenizers import Tokenizer
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    sample_texts = [r['text'][:1000] for r in records[:100]]
    encodings = tokenizer.encode_batch(sample_texts)
    actual_tokens = sum(len(e.ids) for e in encodings)
    sample_chars = sum(len(t) for t in sample_texts)
    chars_per_token = sample_chars / actual_tokens
    print(f"\nActual chars/token:    {chars_per_token:.2f} (from tokenizer)")
    print(f"Corrected token est:   {int(total_chars / chars_per_token):,} for full dataset")
else:
    print("\nTokenizer not trained yet — run tokenizer stage first for accurate token counts")

## 7. Sample documents per source

In [ ]:
rng = random.Random(42)

for source in ['wikipedia', 'code', 'common_crawl']:
    source_records = [r for r in records if r.get('source') == source]
    if not source_records:
        continue
    sample = rng.choice(source_records)
    print(f"{'='*60}")
    print(f"SOURCE: {source.upper()}")
    if 'title' in sample:
        print(f"TITLE:  {sample['title']}")
    if 'url' in sample:
        print(f"URL:    {sample['url']}")
    if 'language' in sample:
        print(f"LANG:   {sample['language']}")
    print(f"CHARS:  {len(sample['text']):,}")
    print(f"TEXT:")
    print(sample['text'][:500])
    print("..." if len(sample['text']) > 500 else "")
    print()

## 8. Word frequency analysis

In [ ]:
import re

def get_words(text, n=1000):
    words = re.findall(r'\b[a-z]{3,}\b', text.lower())
    return words[:n]

# Top words per source
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
source_colors = {'wikipedia': '#4C72B0', 'code': '#55A868', 'common_crawl': '#DD8452'}

for ax, source in zip(axes, ['wikipedia', 'code', 'common_crawl']):
    source_records = [r for r in records if r.get('source') == source][:500]
    if not source_records:
        continue
    all_words = []
    for r in source_records:
        all_words.extend(get_words(r['text']))

    # Filter common stop words for visualization
    stopwords = {'the', 'and', 'for', 'that', 'this', 'with', 'from', 'are', 'was', 'have'}
    counter = Counter(w for w in all_words if w not in stopwords)
    top_words = counter.most_common(15)

    words, freqs = zip(*top_words)
    ax.barh(words[::-1], freqs[::-1], color=source_colors[source], alpha=0.85)
    ax.set_title(f'{source}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Frequency')

fig.suptitle('Top Words by Source (stop words removed)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Chinchilla token targets

In [ ]:
targets = [
    {'model': 'slm-125m', 'params': 125e6, 'chinchilla': 2.5e9, 'planned': 3e9},
    {'model': 'slm-350m', 'params': 350e6, 'chinchilla': 7e9,   'planned': 10e9},
    {'model': 'slm-1b',   'params': 1e9,   'chinchilla': 20e9,  'planned': 25e9},
]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(targets))
width = 0.35

chinchilla = [t['chinchilla'] / 1e9 for t in targets]
planned = [t['planned'] / 1e9 for t in targets]

bars1 = ax.bar(x - width/2, chinchilla, width, label='Chinchilla optimal', color='#4C72B0', alpha=0.85)
bars2 = ax.bar(x + width/2, planned, width, label='SLM planned', color='#DD8452', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([t['model'] for t in targets])
ax.set_ylabel('Tokens (billions)')
ax.set_title('Chinchilla Optimal vs SLM Planned Token Targets', fontsize=12, fontweight='bold')
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}B', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}B', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nChinchilla scaling law: optimal tokens ≈ 20 × parameters")
print("SLM targets include a ~25% buffer above Chinchilla optimal.")